## 1. Import Required Libraries

# MQTT to CSV Logger

This notebook subscribes to MQTT telemetry topics and appends readings to a CSV file. It supports multiple transport protocols (TCP, WebSockets) and optional Supabase integration.

**Installation:**
```
pip install paho-mqtt python-dotenv requests
```

**Configuration:**
Environment variables needed:
- `MQTT_URL`: MQTT broker URL (e.g., mqtt://host:1883 or mqtts://host:8883)
- `MQTT_USERNAME`: MQTT username
- `MQTT_PASSWORD`: MQTT password
- `MQTT_TOPIC_FILTER`: Topics to subscribe to (default: energy/+/+/telemetry)
- `SUPABASE_URL`: (optional) Supabase URL for data forwarding
- `SUPABASE_KEY`: (optional) Supabase API key

In [1]:
from __future__ import annotations
import os
import csv
import json
import time
import signal
import argparse
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse

try:
    import paho.mqtt.client as mqtt
    from paho.mqtt.enums import CallbackAPIVersion
except Exception:
    raise SystemExit("Please install required package: pip install paho-mqtt")

try:
    import requests
except Exception:
    requests = None

try:
    from dotenv import load_dotenv
except ImportError:
    print('Warning: python-dotenv not installed. Install with: pip install python-dotenv')
    print('Falling back to system environment variables only.')
    load_dotenv = None

## 2. Set Environment Variables

In [2]:
# ── ESP32 BMS MQTT Configuration (from config.h) ──
# HiveMQ Cloud broker with TLS on port 8883
MQTT_URL      = 'mqtts://0d34f5789e1e4a669367abfe5bd45b15.s1.eu.hivemq.cloud:8883'
MQTT_USERNAME = 'battery'
MQTT_PASSWORD = 'Batterybms80'
MQTT_TOPIC_FILTER = 'battery/data'   # ESP32 publishes here every 5 seconds

# Optional: Supabase (leave None to disable)
SUPABASE_URL = None
SUPABASE_KEY = None

print("ESP32 BMS MQTT Configuration:")
print(f"  Broker : {MQTT_URL}")
print(f"  User   : {MQTT_USERNAME}")
print(f"  Topic  : {MQTT_TOPIC_FILTER}")


Loaded environment from: server\.env

Environment Configuration:
  MQTT_URL: mqtts://0d34f5789e1e4a669367abfe5bd45b15.s1.eu.hivemq.cloud:8883
  MQTT_USERNAME: battery
  MQTT_PASSWORD: ************
  MQTT_TOPIC_FILTER: battery/data
  SUPABASE_URL: Not set


## 3. Define MQTT Configuration and Constants

In [ ]:
# CSV field names – matches ESP32 JSON payload from mqtt_manager.cpp
FIELDNAMES = [
    'ts', 'ts_iso', 'topic', 'device_type', 'device_id',
    'voltage', 'shunt_mV', 'current', 'power',
    'temperature',
    'soc_percent', 'soh_percent', 'uptime_ms', 'raw_payload'
]

# Global flag for graceful shutdown
stop_requested = False

def signal_handler(sig, frame):
    """Handle SIGINT and SIGTERM for graceful shutdown"""
    global stop_requested
    stop_requested = True
    print("\nShutdown requested...")

# Register signal handlers
signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)


<Handlers.SIG_DFL: 0>

## 4. Parse Broker URL

In [4]:
def parse_broker_url(url: str):
    """
    Parse MQTT broker URL
    
    Returns:
        tuple: (host, port, scheme, path)
    
    Example:
        parse_broker_url('mqtt://localhost:1883') -> ('localhost', 1883, 'mqtt', '')
        parse_broker_url('mqtts://broker.example.com:8883') -> ('broker.example.com', 8883, 'mqtts', '')
    """
    if not url:
        return None, None, None, None
    
    p = urlparse(url)
    scheme = p.scheme
    host = p.hostname
    port = p.port
    path = p.path or ''
    
    return host, port, scheme, path


# Test the parse_broker_url function
test_url = "mqtts://broker.hivemq.cloud:8883"
host, port, scheme, path = parse_broker_url(test_url)
print(f"Parsed URL: {test_url}")
print(f"  Host: {host}, Port: {port}, Scheme: {scheme}, Path: {path}")

Parsed URL: mqtts://broker.hivemq.cloud:8883
  Host: broker.hivemq.cloud, Port: 8883, Scheme: mqtts, Path: 


## 5. Implement MQTTToCSV Class

In [ ]:
class MQTTToCSV:
    """
    MQTT subscriber that logs ESP32 BMS telemetry to CSV.

    Supports TCP (mqtt/mqtts) and WebSocket (ws/wss) transports,
    TLS encryption, and optional Supabase forwarding.
    """

    def __init__(self, broker, port, scheme, path,
                 username, password, topic, outfile,
                 supabase_url=None, supabase_key=None):
        self.broker = broker
        self.port = port
        self.scheme = scheme
        self.path = path or ''
        self.username = username
        self.password = password
        self.topic = topic
        self.outfile = Path(outfile)
        self.supabase_url = supabase_url
        self.supabase_key = supabase_key
        self.msg_count = 0

        # Choose transport
        transport = 'websockets' if scheme in ('ws', 'wss') else 'tcp'
        self.client = mqtt.Client(
            callback_api_version=CallbackAPIVersion.VERSION2,
            transport=transport
        )
        self.client.on_connect = self.on_connect
        self.client.on_message = self.on_message

        if username:
            self.client.username_pw_set(username, password)

        # TLS for mqtts / wss / port 8883 / 8884
        if scheme in ('mqtts', 'wss') or port in (8883, 8884):
            try:
                self.client.tls_set()
                self.client.tls_insecure_set(True)   # skip CA check (dev)
            except Exception as e:
                print(f"TLS setup warning: {e}")

        if scheme in ('ws', 'wss') and self.path:
            try:
                self.client.ws_set_options(path=self.path)
            except Exception as e:
                print(f"WebSocket path warning: {e}")

        # Create CSV with header if it doesn't exist yet
        if not self.outfile.exists():
            with self.outfile.open('w', newline='', encoding='utf-8') as f:
                csv.DictWriter(f, fieldnames=FIELDNAMES).writeheader()
            print(f"Created CSV: {self.outfile}")

    # ── callbacks ────────────────────────────────────────────
    def on_connect(self, client, userdata, flags, reason_code, properties=None):
        print(f'MQTT connected  (rc={reason_code})')
        client.subscribe(self.topic)
        print(f'Subscribed to: {self.topic}')

    def on_message(self, client, userdata, msg):
        try:
            raw = msg.payload.decode('utf-8')
            try:
                p = json.loads(raw)
            except Exception:
                p = {}

            ts = int(time.time() * 1000)
            ts_iso = time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(ts / 1000))

            # Detect device info from topic  (battery/data → battery / esp32_bms)
            parts = msg.topic.split('/')
            if parts[0] == 'battery':
                device_type, device_id = 'battery', 'esp32_bms'
            elif len(parts) >= 3 and parts[0] == 'energy':
                device_type, device_id = parts[1], parts[2]
            else:
                device_type, device_id = parts[0], '/'.join(parts[1:])

            row = {
                'ts':          ts,
                'ts_iso':      ts_iso,
                'topic':       msg.topic,
                'device_type': device_type,
                'device_id':   device_id,
                'voltage':     p.get('bus_V',        p.get('voltage', '')),
                'shunt_mV':    p.get('shunt_mV',     ''),
                'current':     p.get('current_A',    p.get('current', '')),
                'power':       p.get('power_W',      p.get('power', '')),
                'temperature': p.get('temperature',  p.get('temp_c', '')),
                'soc_percent': p.get('soc_percent',  ''),
                'soh_percent': p.get('soh_percent',  ''),
                'uptime_ms':   p.get('uptime_ms',    ''),
                'raw_payload': raw
            }

            with self.outfile.open('a', newline='', encoding='utf-8') as f:
                csv.DictWriter(f, fieldnames=FIELDNAMES).writerow(row)

            self.msg_count += 1
            print(f'[{self.msg_count}] V={row["voltage"]}  I={row["current"]}  '
                  f'P={row["power"]}  T={row["temperature"]}°C  '
                  f'SoC={row["soc_percent"]}%  → {self.outfile.name}')

            # Optional Supabase forwarding
            if self.supabase_url and self.supabase_key:
                try:
                    self._send_to_supabase(p, row, msg.topic)
                except Exception as e:
                    print(f'Supabase upload failed: {e}')

        except Exception as e:
            print(f'Error processing message: {e}')

    # ── Supabase ─────────────────────────────────────────────
    def _send_to_supabase(self, p, row, topic):
        if requests is None:
            raise RuntimeError('pip install requests')
        data = {
            'ts': row['ts'], 'ts_iso': row['ts_iso'],
            'topic': topic,
            'device_type': row['device_type'],
            'device_id':   row['device_id'],
            'voltage':   p.get('bus_V')      or p.get('voltage'),
            'current':   p.get('current_A')  or p.get('current'),
            'power':     p.get('power_W')    or p.get('power'),
            'temperature': p.get('temperature'),
            'soc':       p.get('soc_percent'),
            'soh':       p.get('soh_percent'),
            'uptime_ms': p.get('uptime_ms'),
            'raw_payload': row['raw_payload']
        }
        url = self.supabase_url.rstrip('/') + '/rest/v1/telemetry'
        headers = {
            'apikey': self.supabase_key,
            'Authorization': f'Bearer {self.supabase_key}',
            'Content-Type': 'application/json',
            'Prefer': 'return=representation'
        }
        resp = requests.post(url, headers=headers, json=[data], timeout=10)
        if resp.status_code not in (200, 201):
            raise RuntimeError(f'{resp.status_code} {resp.text}')

    # ── main loop ────────────────────────────────────────────
    def run(self):
        global stop_requested
        if not self.broker:
            raise RuntimeError('No broker host')
        port = self.port or (8883 if self.scheme in ('mqtts', 'wss') else 1883)
        print(f'Connecting to {self.broker}:{port}  (scheme={self.scheme})')
        self.client.connect(self.broker, port, keepalive=60)
        try:
            while not stop_requested:
                self.client.loop(timeout=1.0)
        except KeyboardInterrupt:
            stop_requested = True
            print("\nInterrupted – shutting down…")
        finally:
            try:
                self.client.disconnect()
                print("Disconnected from broker")
            except Exception:
                pass


## 6. Execute Main Logic

In [ ]:
# ── Configuration (uses ESP32 BMS values set in Cell 4) ──
config = {
    'broker':  MQTT_URL,                       # mqtts://...hivemq.cloud:8883
    'topic':   MQTT_TOPIC_FILTER,              # battery/data
    'username': MQTT_USERNAME,                 # battery
    'password': MQTT_PASSWORD,                 # Batterybms80
    'outfile': 'Cycle 01/b1_discharge.csv',    # output CSV path
    'supabase_url': SUPABASE_URL,
    'supabase_key': SUPABASE_KEY
}

print("=== MQTT → CSV Logger ===")
print(f"Broker : {config['broker']}")
print(f"Topic  : {config['topic']}")
print(f"Output : {config['outfile']}")
print(f"Supabase: {'Enabled' if config['supabase_url'] else 'Disabled'}")
print("=========================\n")



=== MQTT to CSV Logger Configuration ===
Broker: mqtts://0d34f5789e1e4a669367abfe5bd45b15.s1.eu.hivemq.cloud:8883
Topic: battery/data
Output File: Cycle 01/b1_discharge.csv
Supabase Integration: Disabled



In [7]:
# Parse broker URL and create service
broker_url = config['broker']
if '://' in broker_url:
    host, port, scheme, path = parse_broker_url(broker_url)
else:
    host = broker_url
    port = None
    scheme = None
    path = ''

# Ensure output directory exists
output_path = Path(config['outfile'])
output_path.parent.mkdir(parents=True, exist_ok=True)

# Create and run the MQTT to CSV service
service = MQTTToCSV(
    broker=host,
    port=port,
    scheme=scheme,
    path=path,
    username=config['username'],
    password=config['password'],
    topic=config['topic'],
    outfile=config['outfile'],
    supabase_url=config['supabase_url'],
    supabase_key=config['supabase_key']
)

print("Starting MQTT subscriber...")
print("(Press Ctrl+C to stop)\n")

try:
    service.run()
except KeyboardInterrupt:
    stop_requested = True
    print("\n\nShutdown requested (Ctrl+C). Waiting for cleanup...")
    try:
        service.client.disconnect()
        print("Disconnected from broker")
    except Exception:
        pass
except Exception as e:
    print(f"Error: {e}")

Starting MQTT subscriber...
(Press Ctrl+C to stop)

Connecting to MQTT broker 0d34f5789e1e4a669367abfe5bd45b15.s1.eu.hivemq.cloud:8883 (scheme=mqtts, path=)
Error: [Errno 11001] getaddrinfo failed


## 7. Data Analysis (Optional)

In [ ]:
# Read and display collected CSV data
import pandas as pd

outfile = Path(config['outfile'])
if outfile.exists():
    df = pd.read_csv(outfile)
    print(f"Records : {len(df)}")
    if len(df):
        print(f"Range   : {df['ts_iso'].min()} → {df['ts_iso'].max()}")
        print(f"Devices : {df['device_id'].unique().tolist()}")
        # Show numeric summary for key columns
        num_cols = ['voltage', 'current', 'power', 'temperature', 'soc_percent']
        present = [c for c in num_cols if c in df.columns]
        if present:
            print(f"\n{df[present].astype(float, errors='ignore').describe().round(3)}")
        print(f"\nLast 5 rows:")
        print(df.tail())
else:
    print(f"CSV not found: {outfile}")
    print("Run the logger cells first to collect data from the ESP32.")


Data Summary (12 records):
Date Range: 2026-02-28T18:10:36 to 2026-02-28T18:11:31

Devices: ['esp32_bms']

Topics: ['battery/data']

Latest 5 records:
               ts               ts_iso         topic device_type  device_id  \
7   1772280671029  2026-02-28T18:11:11  battery/data     battery  esp32_bms   
8   1772280676034  2026-02-28T18:11:16  battery/data     battery  esp32_bms   
9   1772280681036  2026-02-28T18:11:21  battery/data     battery  esp32_bms   
10  1772280686029  2026-02-28T18:11:26  battery/data     battery  esp32_bms   
11  1772280691034  2026-02-28T18:11:31  battery/data     battery  esp32_bms   

    voltage  shunt_mV  current  power  soc_percent  soh_percent  uptime_ms  \
7       NaN       0.0      NaN    NaN          0.0          0.0    1202180   
8       NaN       0.0      NaN    NaN          0.0          0.0    1207181   
9       NaN       0.0      NaN    NaN          0.0          0.0    1212182   
10      NaN       0.0      NaN    NaN          0.0          0.